# RockyEye — stages 01–02 on Kaggle: gj3929b

4 real eclipses of GJ 3929 b: raw JWST ramps → stamps. Runs the official `jwst` pipeline (Detector1 + Image2) on
25 raw segments (32 GB, attached, never copied) and writes one small **stamp file** per
segment: an 81×81 px cut-out around the star for every integration. Everything after that
(photometry, fitting, physics) runs on the laptop.

**Before running**

1. *Add Input* → add the dataset `stsci/rocky-worlds-gj-3929b-observations`.
2. *Settings* → **Internet: On** (for `pip install jwst` and the CRDS reference files). No GPU.
3. *Save Version* → **Save & Run All (Commit)**. A commit runs in the background and keeps
   `/kaggle/working` as the notebook's output.

**How it protects a long run**

* Each segment's stamp is written to `/kaggle/working/stamps/` as soon as that segment is done,
  and `stamps_index.json` is rewritten after every segment.
* A segment that fails is logged and skipped; the run carries on.
* It stops by itself after `MAX_HOURS` = 9.0 h (Kaggle's limit is 12 h), so the commit always
  finishes and the output is always saved.
* **To continue or retry**: start a new version with the previous version's output attached as an
  input (*Add Input → Your Work → Notebook Output*). Finished stamps are copied over first and
  skipped; only missing segments are computed.

Per segment: read the attached `*_uncal.fits` → 50-integration chunks in `/kaggle/temp` → stage 1 →
stage 2 → cut the stamp → delete chunk, ramp, rateints and calints files → next segment.

**Afterwards**: *Output → Download All*, then on the laptop:
`make import FROM=~/Downloads/<the zip or folder>`.

In [ ]:
# ---- parameters ---------------------------------------------------------------
TARGET = "gj3929b"          # config name (one dataset per notebook)
VISITS = None              # e.g. ["eclipse1", "eclipse2"] to do a subset; None = all
MAX_HOURS = 9.0            # stop cleanly well before Kaggle's 12 h session limit
JWST_VERSION = "3.0.0"     # same version as the laptop environment

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"jwst=={JWST_VERSION}", "pyyaml"], check=True)
import jwst
print("jwst", jwst.__version__)

## Pipeline code

Written out verbatim from the RockyEye repository (`scripts/build_kaggle_notebook.py` generates these cells).

In [ ]:
import os
for d in ('rockyeye', 'configs'):
    os.makedirs(f'/kaggle/working/rockyeye_src/' + d, exist_ok=True)

In [ ]:
%%writefile /kaggle/working/rockyeye_src/rockyeye/__init__.py
# package marker

In [ ]:
%%writefile /kaggle/working/rockyeye_src/rockyeye/common.py
"""Shared plumbing: config loading, paths, caching, logging.

Nothing scientific lives here. Every stage script does the same three things:
load a YAML config, work out where its products go, and skip work whose cached
output already matches the config that produced it.
"""
from __future__ import annotations

import hashlib
import json
import logging
import os
import time
from pathlib import Path

import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[1]

log = logging.getLogger("rockyeye")


def setup_logging(logfile: str | Path | None = None, level=logging.INFO) -> None:
    """Log to the terminal and (optionally) append to a file, so overnight runs leave a trail."""
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S")
    root = logging.getLogger("rockyeye")
    root.setLevel(level)
    root.handlers.clear()
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    root.addHandler(sh)
    if logfile is not None:
        Path(logfile).parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(logfile)
        fh.setFormatter(fmt)
        root.addHandler(fh)


def _deep_update(base: dict, new: dict) -> dict:
    out = dict(base)
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_update(out[k], v)
        else:
            out[k] = v
    return out


def load_config(path: str | Path) -> dict:
    """Read a YAML config. `extends: other.yaml` layers this file over another one."""
    path = Path(path)
    if not path.is_absolute() and not path.exists():
        path = PROJECT_ROOT / path
    with open(path) as f:
        cfg = yaml.safe_load(f)
    if "extends" in cfg:
        parent = load_config(path.parent / cfg.pop("extends"))
        cfg = _deep_update(parent, cfg)
    cfg["_config_path"] = str(path)
    return cfg


def resolve(p: str | Path) -> Path:
    """Paths in configs are relative to projrocky/."""
    p = Path(p)
    return p if p.is_absolute() else PROJECT_ROOT / p


def data_dir(cfg: dict) -> Path:
    return resolve(cfg["paths"]["data"])


def product_dir(cfg: dict, stage: str = "") -> Path:
    d = resolve(cfg["paths"]["products"]) / stage
    d.mkdir(parents=True, exist_ok=True)
    return d


def config_hash(*sections) -> str:
    """Short, stable fingerprint of the config sections a product depends on."""
    blob = json.dumps(sections, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:12]


def is_cached(path: Path, key: str) -> bool:
    """A product is fresh if it exists and its sidecar records the same config hash."""
    side = Path(str(path) + ".key")
    return path.exists() and side.exists() and side.read_text().strip() == key


def mark_cached(path: Path, key: str) -> None:
    Path(str(path) + ".key").write_text(key)


def atomic_write_json(path: Path, obj) -> None:
    """Write JSON via a temp file so a crash never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    os.replace(tmp, path)


def _json_default(o):
    import numpy as np

    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def read_json(path: Path):
    with open(path) as f:
        return json.load(f)


def free_gb(path: Path) -> float:
    import shutil

    path = Path(path)
    while not path.exists():
        path = path.parent
    return shutil.disk_usage(path).free / 1e9


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.dt = time.time() - self.t0

In [ ]:
%%writefile /kaggle/working/rockyeye_src/rockyeye/s01_fetch.py
"""Ingredient 01 — data fetcher with a dry-run mode.

Two sources are supported, both without any login:

* ``kaggle``: the Rocky Worlds Data Challenge datasets, fetched with Kaggle's
  official ``kagglehub`` client (public datasets need no credentials). A
  ``manifest.csv`` lists every ``*_uncal.fits`` file with its size and SHA-256.
* ``mast``: the public JWST archive, listed with astroquery and downloaded
  from the MAST file endpoint.

Design rules (from the spec):
  1. Never download before the size is known and printed.
  2. ``dry_run=True`` prints the plan and touches nothing.
  3. Downloads resume (HTTP Range into a ``.part`` file) and are verified.
  4. Refuse to start a file that would push free disk below a safety floor.
"""
from __future__ import annotations

import csv
import hashlib
import os
import shutil
import time
import urllib.parse
from dataclasses import dataclass, asdict
from pathlib import Path

import requests

from .common import data_dir, free_gb, log, resolve

MAST_FILE = "https://mast.stsci.edu/api/v0.1/Download/file"


@dataclass
class RemoteFile:
    """One raw file we might download."""

    visit: str          # e.g. "eclipse3"
    segment: int        # 1-based segment number within the visit
    filename: str
    url: str
    size_bytes: int
    sha256: str | None  # None when the archive gives no checksum
    local_path: str

    @property
    def size_gb(self) -> float:
        return self.size_bytes / 1e9


# ----------------------------------------------------------------- listing ---

def _kagglehub(cfg: dict):
    """Kaggle's official client, with its download cache kept inside the project's data folder."""
    os.environ.setdefault("KAGGLEHUB_CACHE", str(resolve(cfg["paths"].get("kaggle_cache", "data/kagglehub_cache"))))
    import kagglehub

    return kagglehub


def _kaggle_manifest(cfg: dict) -> Path:
    """Fetch (once) the tiny manifest.csv that describes the dataset."""
    src = cfg["source"]
    root = os.environ.get("ROCKYEYE_INPUT_ROOT") or src.get("local_root")
    if root and (Path(root) / "manifest.csv").exists():
        return Path(root) / "manifest.csv"
    path = resolve(src["manifest"])
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        got = _kagglehub(cfg).dataset_download(src["dataset"], path="manifest.csv")
        shutil.copyfile(got, path)
        log.info("fetched manifest (%d bytes) -> %s", path.stat().st_size, path)
    return path


def _list_kaggle(cfg: dict) -> list[RemoteFile]:
    """Files of a Kaggle dataset, from its manifest.

    Two optional config keys under ``source``:
      local_root        the dataset is already on disk here (a Kaggle notebook's
                        /kaggle/input/<dataset>): nothing is downloaded or deleted.
      visit_from_obsnum {obs number: visit name}: rename by JWST observation
                        number, used to split GJ 3929 b's eclipse 4 into 4a/4b.
    """
    src = cfg["source"]
    raw = data_dir(cfg) / "raw"
    local_root = os.environ.get("ROCKYEYE_INPUT_ROOT") or src.get("local_root")
    by_obs = {int(k): v for k, v in (src.get("visit_from_obsnum") or {}).items()}
    out = []
    with open(_kaggle_manifest(cfg)) as f:
        for row in csv.DictReader(f):
            name = row["filename"]
            visit = row["eclipse"]
            if by_obs and name.startswith("jw") and int(name[7:10]) in by_obs:
                visit = by_obs[int(name[7:10])]
            local = Path(local_root) / row["relative_path"] if local_root else raw / visit / name
            out.append(RemoteFile(
                visit=visit,
                segment=int(row["segment"]),
                filename=name,
                url=f"kaggle://{src['dataset']}/{row['relative_path']}",
                size_bytes=int(row["size_bytes"]),
                sha256=row["sha256"],
                local_path=str(local),
            ))
    return out


def is_input_file(cfg: dict, rf: RemoteFile) -> bool:
    """True if the raw file lives in a read-only input folder that we must never delete from."""
    root = os.environ.get("ROCKYEYE_INPUT_ROOT") or cfg["source"].get("local_root")
    return bool(root) and str(rf.local_path).startswith(str(root))


def _list_mast(cfg: dict) -> list[RemoteFile]:
    """List uncal files of a JWST program on MAST. Visits come from ``source.visits``.

    ``source.visits`` maps our visit names to lists of JWST observation numbers,
    e.g. ``eclipse4: [4, 5]`` because GJ 3929 b's 4th eclipse was split in two.
    The listing is cached to JSON so later dry-runs work offline.
    """
    import json

    src = cfg["source"]
    cache = data_dir(cfg) / "mast_listing.json"
    if cache.exists():
        rows = json.loads(cache.read_text())
    else:
        from astroquery.mast import Observations

        obs = Observations.query_criteria(
            obs_collection="JWST", proposal_id=str(src["program"]),
            instrument_name="MIRI/IMAGE", filters=src.get("filter", "F1500W"))
        prods = Observations.get_product_list(obs)
        keep = Observations.filter_products(prods, productSubGroupDescription="UNCAL")
        rows = sorted({(str(p["productFilename"]), int(p["size"]), str(p["dataURI"])) for p in keep})
        rows = [list(r) for r in rows]
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(json.dumps(rows, indent=1))
    obs_to_visit = {int(o): v for v, obs_list in src["visits"].items() for o in obs_list}
    raw = data_dir(cfg) / "raw"
    out = []
    for fname, size, uri in rows:
        # jw<ppppp><ooo><vvv>_<gg><s><aa>_<eeeee>-seg<NNN>_mirimage_uncal.fits
        obsnum = int(fname[7:10])
        if obsnum not in obs_to_visit or "-seg" not in fname:
            continue
        seg = int(fname.split("-seg")[1][:3])
        visit = obs_to_visit[obsnum]
        out.append(RemoteFile(
            visit=visit, segment=seg, filename=fname,
            url=f"{MAST_FILE}?uri={urllib.parse.quote(uri, safe='')}",
            size_bytes=size, sha256=None,
            local_path=str(raw / visit / fname)))
    return out


def _list_synthetic(cfg: dict) -> list[RemoteFile]:
    """Placeholder "raw files" for the synthetic test bench (see synth.py): nothing to download."""
    n_seg = cfg["source"].get("segments_per_visit", 3)
    return [RemoteFile(visit=v, segment=s, filename=f"synthetic-{v}-seg{s:03d}_mirimage_uncal.fits",
                       url="synthetic://", size_bytes=0, sha256=None, local_path="")
            for v in cfg["visits"] for s in range(1, n_seg + 1)]


def list_files(cfg: dict, visits: list[str] | None = None) -> list[RemoteFile]:
    """All raw files for this config, ordered by visit then time, optionally filtered."""
    kind = cfg["source"]["kind"]
    files = {"kaggle": _list_kaggle, "mast": _list_mast, "synthetic": _list_synthetic}[kind](cfg)
    if visits:
        files = [f for f in files if f.visit in visits]
    # filename order == time order within a visit (obs number, then segment)
    files.sort(key=lambda f: (f.visit, f.filename))
    return files


# ---------------------------------------------------------------- planning ---

def plan(cfg: dict, visits: list[str] | None = None) -> dict:
    """Work out what a download would involve, without downloading anything."""
    files = list_files(cfg, visits)
    todo = [f for f in files if not Path(f.local_path).exists()]
    per_visit: dict[str, dict] = {}
    for f in files:
        d = per_visit.setdefault(f.visit, {"n_files": 0, "gb": 0.0, "gb_missing": 0.0})
        d["n_files"] += 1
        d["gb"] += f.size_gb
        if f in todo:
            d["gb_missing"] += f.size_gb
    return {
        "n_files": len(files),
        "total_gb": sum(f.size_gb for f in files),
        "missing_gb": sum(f.size_gb for f in todo),
        "largest_file_gb": max((f.size_gb for f in files), default=0.0),
        "free_gb": free_gb(data_dir(cfg)),
        "per_visit": per_visit,
        "files": [asdict(f) for f in files],
    }


def format_plan(p: dict, streaming: bool = True) -> str:
    """Human-readable dry-run report."""
    lines = [f"{'visit':<10} {'files':>5} {'size GB':>8} {'to fetch GB':>12}"]
    for v, d in p["per_visit"].items():
        lines.append(f"{v:<10} {d['n_files']:>5} {d['gb']:>8.2f} {d['gb_missing']:>12.2f}")
    lines.append(f"{'TOTAL':<10} {p['n_files']:>5} {p['total_gb']:>8.2f} {p['missing_gb']:>12.2f}")
    lines.append(f"free disk: {p['free_gb']:.1f} GB; largest single file: {p['largest_file_gb']:.2f} GB")
    if streaming:
        lines.append("streaming mode: one raw file on disk at a time "
                     f"(peak raw footprint ~{p['largest_file_gb']:.1f} GB, not {p['total_gb']:.0f} GB)")
    elif p["missing_gb"] > p["free_gb"]:
        lines.append("WARNING: the full set does NOT fit on this disk; use streaming mode")
    return "\n".join(lines)


# ------------------------------------------------------------- downloading ---

def sha256_of(path: str | Path, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def download(rf: RemoteFile, min_free_gb: float = 8.0, dry_run: bool = False,
             max_retries: int = 8, verify: bool = True) -> Path:
    """Download one file, resuming a partial ``.part`` if present.

    Returns the final path. With ``dry_run`` it only logs what it would do.
    """
    dest = Path(rf.local_path)
    if dest.exists() and dest.stat().st_size == rf.size_bytes:
        return dest
    part = dest.with_suffix(dest.suffix + ".part")
    have = part.stat().st_size if part.exists() else 0
    need_gb = (rf.size_bytes - have) / 1e9
    if dry_run:
        log.info("[dry-run] would download %s (%.2f GB, %.2f GB remaining)",
                 rf.filename, rf.size_gb, need_gb)
        return dest
    if free_gb(dest.parent) - need_gb < min_free_gb:
        raise OSError(f"refusing to download {rf.filename}: needs {need_gb:.2f} GB and would "
                      f"leave less than {min_free_gb} GB free ({free_gb(dest.parent):.1f} GB free now)")
    dest.parent.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("KAGGLEHUB_CACHE", str(resolve("data/kagglehub_cache")))

    for attempt in range(1, max_retries + 1):
        have = part.stat().st_size if part.exists() else 0
        if have >= rf.size_bytes:
            break
        try:
            if rf.url.startswith("kaggle://"):
                _fetch_kaggle(rf, part)
            else:
                _fetch_http(rf, part, have)
        except Exception as e:   # network errors of any flavour: wait and resume
            wait = min(300, 10 * 2 ** (attempt - 1))
            log.warning("download %s interrupted (%s: %s); retry %d/%d in %ds", rf.filename,
                        type(e).__name__, str(e)[:80], attempt, max_retries, wait)
            time.sleep(wait)
    size = part.stat().st_size if part.exists() else 0
    if size != rf.size_bytes:
        raise OSError(f"{rf.filename}: got {size} bytes, expected {rf.size_bytes}")
    if verify and rf.sha256:
        digest = sha256_of(part)
        if digest != rf.sha256:
            part.unlink()
            raise OSError(f"{rf.filename}: SHA-256 mismatch (file deleted, rerun to refetch)")
    os.replace(part, dest)
    return dest


def _fetch_http(rf: RemoteFile, part: Path, have: int) -> None:
    """Plain HTTPS download that continues a partial file with a Range request (used for MAST)."""
    headers = {"Range": f"bytes={have}-"} if have else {}
    with requests.get(rf.url, headers=headers, stream=True, timeout=(30, 120)) as r:
        if have and r.status_code == 200:   # server ignored Range: start over
            have = 0
            part.unlink(missing_ok=True)
        r.raise_for_status()
        with open(part, "ab" if have else "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 22):
                f.write(chunk)


def _fetch_kaggle(rf: RemoteFile, part: Path) -> None:
    """Download one dataset file with kagglehub, then move it out of kagglehub's cache.

    kagglehub keeps its own partial file in the cache and resumes it, so a
    retry after a dropped connection does not start from zero.
    """
    import kagglehub

    dataset, _, rel = rf.url[len("kaggle://"):].partition("/")
    owner_slug = dataset + "/" + rel.split("/", 1)[0]
    rel_path = rel.split("/", 1)[1]
    got = Path(kagglehub.dataset_download(owner_slug, path=rel_path))
    shutil.move(str(got), part)


def fetch_small(url: str, dest: Path, max_mb: float = 200.0, dry_run: bool = False) -> Path:
    """Fetch a small auxiliary file (benchmark light curves, filter curves), size-checked first."""
    dest = Path(dest)
    if dest.exists():
        return dest
    head = requests.head(url, allow_redirects=True, timeout=60)
    size = int(head.headers.get("content-length", 0))
    if size / 1e6 > max_mb:
        raise OSError(f"{url} is {size/1e6:.0f} MB, above the {max_mb} MB limit for auxiliary files")
    if dry_run:
        log.info("[dry-run] would fetch %s (%.2f MB)", url, size / 1e6)
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    tmp = dest.with_suffix(dest.suffix + ".part")
    tmp.write_bytes(r.content)
    os.replace(tmp, dest)
    log.info("fetched %s (%.2f MB)", dest.name, len(r.content) / 1e6)
    return dest

In [ ]:
%%writefile /kaggle/working/rockyeye_src/rockyeye/s02_calibrate.py
"""Ingredient 02 — stage 1-2 runner (official ``jwst`` pipeline, config-driven).

Stage 1 (``Detector1Pipeline``) turns raw up-the-ramp reads into a count rate
per pixel per integration ("rateints"). Stage 2 (``Image2Pipeline``) flat-fields
those ("calints"). We do not re-implement either: the config only chooses which
official steps run and with which parameters.

Disk is the constraint on a laptop: the raw simulated set is 199 GB. So the
runner *streams*: for each segment it downloads the raw file, calibrates it,
keeps a small cut-out ("stamp") around the star, and deletes the big files
before touching the next segment. The stamp is all later stages need, and it
lets photometry be redone with any aperture in seconds.

Resumable by construction: a segment whose stamp exists (and whose config hash
matches) is skipped, so an interrupted overnight run continues where it stopped.
"""
from __future__ import annotations

import os
import shutil
from pathlib import Path

import numpy as np

from .common import (Timer, atomic_write_json, config_hash, data_dir, is_cached, log,
                     mark_cached, product_dir, read_json, resolve)
from . import s01_fetch


def setup_crds(cfg: dict) -> None:
    """Point the jwst pipeline at a local reference-file cache inside the project."""
    crds = resolve(cfg["paths"].get("crds", "data/crds_cache"))
    crds.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("CRDS_PATH", str(crds))
    os.environ.setdefault("CRDS_SERVER_URL", "https://jwst-crds.stsci.edu")
    if cfg.get("stage1", {}).get("crds_context"):
        os.environ["CRDS_CONTEXT"] = cfg["stage1"]["crds_context"]


def split_uncal(uncal: Path, chunk_ints: int, work: Path) -> list[Path]:
    """Cut a raw segment into files of at most ``chunk_ints`` integrations.

    Why: stage 1 holds several float copies of the whole ramp cube in memory.
    A 2 GB segment (271 integrations x 45 groups x 256 x 256) needs well over
    16 GB that way; 50 integrations at a time needs a few GB. A chunk is
    written exactly like an official segment file: same headers, with the
    INTSTART/INTEND keywords saying which integrations of the exposure it holds.
    """
    from astropy.io import fits

    work.mkdir(parents=True, exist_ok=True)
    out = []
    with fits.open(uncal, memmap=False) as h:          # .section below reads lazily, slice by slice
        n = h["SCI"].header["NAXIS4"]
        first = h[0].header["INTSTART"]
        if not chunk_ints or n <= chunk_ints:
            return [uncal]
        n_chunks = int(np.ceil(n / chunk_ints))
        edges = np.linspace(0, n, n_chunks + 1).round().astype(int)
        for k, (a, b) in enumerate(zip(edges[:-1], edges[1:])):
            path = work / uncal.name.replace("_mirimage_uncal.fits", f"-c{k:02d}_mirimage_uncal.fits")
            hdus = [fits.PrimaryHDU(header=h[0].header.copy())]
            hdus[0].header["INTSTART"] = first + a
            hdus[0].header["INTEND"] = first + b - 1
            for ext in h[1:]:
                if ext.name in ("SCI", "REFOUT"):
                    # .section reads just these integrations and applies the unsigned-16-bit
                    # offset (BZERO), so the chunk is written as proper uint16 again
                    hdu = fits.ImageHDU(data=ext.section[a:b], header=ext.header.copy(), name=ext.name)
                elif ext.name == "GROUP":                      # per-integration (or per-group) rows
                    per = max(len(ext.data) // n, 1)
                    hdu = fits.BinTableHDU(data=ext.data[a * per:b * per], header=ext.header.copy(), name="GROUP")
                else:                                          # INT_TIMES (whole exposure) and ASDF: unchanged
                    hdu = ext.copy()
                hdus.append(hdu)
            fits.HDUList(hdus).writeto(path, overwrite=True)
            out.append(path)
    return out


def run_stage1(uncal: Path, out_dir: Path, steps: dict) -> Path:
    """Detector1Pipeline: uncal ramps -> rateints (DN/s per integration)."""
    from jwst.pipeline import Detector1Pipeline

    out_dir.mkdir(parents=True, exist_ok=True)
    Detector1Pipeline.call(str(uncal), steps=steps or {}, save_results=True, output_dir=str(out_dir),
                           save_calibrated_ramp=False)
    out = out_dir / uncal.name.replace("_uncal.fits", "_rateints.fits")
    if not out.exists():
        raise FileNotFoundError(f"stage 1 did not produce {out}")
    # Only rateints is wanted. Remove the all-integration average and any corrected-ramp
    # file a step may have saved (0.8 GB per chunk): leaving them fills the disk.
    for suffix in ("_rate.fits", "_ramp.fits", "_jump.fits"):
        (out_dir / uncal.name.replace("_uncal.fits", suffix)).unlink(missing_ok=True)
    return out


def run_stage2(rateints: Path, out_dir: Path, steps: dict) -> Path:
    """Image2Pipeline: rateints -> calints (flat-fielded; flux calibration is optional)."""
    from jwst.pipeline import Image2Pipeline

    Image2Pipeline.call(str(rateints), steps=steps or {}, save_results=True, output_dir=str(out_dir))
    out = out_dir / rateints.name.replace("_rateints.fits", "_calints.fits")
    if not out.exists():
        raise FileNotFoundError(f"stage 2 did not produce {out}")
    return out


def find_star(image: np.ndarray, guess: tuple[float, float] | None = None, box: int = 40):
    """Rough star position (x, y) on a median image: brightest 3x3-smoothed pixel.

    The border is ignored, and if ``guess`` is given only a box around it is searched.
    """
    from scipy.ndimage import uniform_filter

    img = np.nan_to_num(image - np.nanmedian(image))
    sm = uniform_filter(img, 3)
    mask = np.zeros_like(sm, dtype=bool)
    ny, nx = sm.shape
    if guess is None:
        mask[8:ny - 8, 8:nx - 8] = True
    else:
        gx, gy = int(round(guess[0])), int(round(guess[1]))
        mask[max(gy - box, 0):gy + box + 1, max(gx - box, 0):gx + box + 1] = True
    sm = np.where(mask, sm, -np.inf)
    iy, ix = np.unravel_index(np.argmax(sm), sm.shape)
    return float(ix), float(iy)


def select_times(number: np.ndarray, t_all: np.ndarray, i0, i1, n: int) -> np.ndarray:
    """Mid-integration times of the ``n`` integrations a file holds.

    The INT_TIMES table may list more integrations than the file contains (a
    whole segment, or the whole exposure). Rows are picked by integration
    number (INTSTART..INTEND), never by position; if the header lacks those
    keywords the table must match the data one-to-one.
    """
    if i0 is not None and i1 is not None:
        t = t_all[(number >= i0) & (number <= i1)]
        if len(t) == n:
            return t
    if len(t_all) == n:
        return t_all
    raise ValueError(f"cannot match {n} integrations to the INT_TIMES table "
                     f"({len(t_all)} rows, numbers {number.min()}..{number.max()}, INTSTART={i0}, INTEND={i1})")


def make_stamp(calints: Path, centre: tuple[float, float] | None, half: int,
               guess: tuple[float, float] | None = None) -> dict:
    """Cut a (2*half+1)^2 stamp around the star out of every integration.

    Returns times (BJD_TDB), the science and error cubes, a bad-pixel mask and
    the stamp's origin in full-frame pixel coordinates.
    """
    from jwst import datamodels

    with datamodels.open(str(calints)) as m:
        sci = np.array(m.data, dtype=np.float32)
        err = np.array(m.err, dtype=np.float32)
        dq = np.array(m.dq)
        t = select_times(np.array(m.int_times["integration_number"]),
                         np.array(m.int_times["int_mid_BJD_TDB"], dtype=float) + 2400000.5,
                         m.meta.exposure.integration_start, m.meta.exposure.integration_end, sci.shape[0])
        unit = m.meta.bunit_data
        pixar_sr = m.meta.photometry.pixelarea_steradians or np.nan
    med = np.nanmedian(sci, axis=0)
    if centre is None:
        centre = find_star(med, guess)
    cx, cy = int(round(centre[0])), int(round(centre[1]))
    ny, nx = med.shape
    y0, y1 = max(cy - half, 0), min(cy + half + 1, ny)
    x0, x1 = max(cx - half, 0), min(cx + half + 1, nx)
    do_not_use = (dq[:, y0:y1, x0:x1] & 1) > 0
    return {"time": t, "sci": sci[:, y0:y1, x0:x1], "err": err[:, y0:y1, x0:x1],
            "bad": do_not_use, "origin": np.array([x0, y0]), "centre": np.array(centre),
            "median_frame": med.astype(np.float32), "unit": str(unit), "pixar_sr": float(pixar_sr)}


def stamp_path(cfg: dict, rf: s01_fetch.RemoteFile) -> Path:
    d = product_dir(cfg, "stage02") / rf.visit
    d.mkdir(parents=True, exist_ok=True)
    return d / rf.filename.replace("_uncal.fits", "_stamp.npz")


def stage_key(cfg: dict) -> str:
    return config_hash(cfg.get("stage1"), cfg.get("stage2"), cfg["stamp"])


def process_segment(cfg: dict, rf: s01_fetch.RemoteFile, dry_run: bool = False) -> Path:
    """Fetch -> stage 1 -> stage 2 -> stamp -> clean up, for one raw segment."""
    out = stamp_path(cfg, rf)
    key = stage_key(cfg)
    if is_cached(out, key):
        return out
    if dry_run:
        have = Path(rf.local_path).exists()
        log.info("[dry-run] %s %s: %s, then stage 1-2 -> %s", rf.visit, rf.filename,
                 "raw already on disk" if have else f"download {rf.size_gb:.2f} GB", out.name)
        return out
    setup_crds(cfg)
    st = cfg["stamp"]
    work = data_dir(cfg) / "work" / rf.visit
    keep = cfg.get("keep", {})
    uncal = s01_fetch.download(rf, min_free_gb=cfg["source"].get("min_free_gb", 8.0))
    # the star position is fixed per visit: measured on the first chunk, reused afterwards
    cfile = product_dir(cfg, "stage02") / rf.visit / "centre.json"
    centre = tuple(read_json(cfile)["centre"]) if cfile.exists() else None
    chunks = split_uncal(uncal, cfg.get("stage1", {}).get("chunk_ints", 50), work)
    t1 = t2 = 0.0
    parts = []
    for chunk in chunks:
        with Timer() as tm:
            rateints = run_stage1(chunk, work, cfg.get("stage1", {}).get("steps", {}))
        t1 += tm.dt
        with Timer() as tm:
            calints = run_stage2(rateints, work, cfg.get("stage2", {}).get("steps", {}))
        t2 += tm.dt
        part = make_stamp(calints, centre, st["half_size"], guess=st.get("guess"))
        if centre is None:
            centre = tuple(float(c) for c in part["centre"])
            atomic_write_json(cfile, {"centre": centre, "from": rf.filename})
        parts.append(part)
        if chunk != uncal:
            chunk.unlink(missing_ok=True)
        for product, kind in ((rateints, "rateints"), (calints, "calints")):
            if keep.get(kind, False):
                dest = data_dir(cfg) / kind / rf.visit
                dest.mkdir(parents=True, exist_ok=True)
                shutil.move(str(product), dest / product.name)
            else:
                product.unlink(missing_ok=True)
    stamp = {k: np.concatenate([p[k] for p in parts]) for k in ("time", "sci", "err", "bad")}
    for k in ("origin", "centre", "median_frame", "unit", "pixar_sr"):
        stamp[k] = parts[0][k]
    tmp = out.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **stamp, filename=rf.filename, visit=rf.visit, segment=rf.segment)
    os.replace(tmp, out)
    mark_cached(out, key)
    if not keep.get("raw", False) and not s01_fetch.is_input_file(cfg, rf):
        uncal.unlink(missing_ok=True)
    log.info("%s %s done: %d ints in %d chunks | stage1 %.0fs, stage2 %.0fs", rf.visit, rf.filename,
             len(stamp["time"]), len(chunks), t1, t2)
    return out


def process(cfg: dict, visits: list[str] | None = None, dry_run: bool = False,
            max_segments: int | None = None, max_hours: float | None = None,
            continue_on_error: bool = False, after_segment=None) -> list[Path]:
    """Stream every segment of the requested visits through stages 1-2.

    Each finished segment is written to disk (atomically) before the next one
    starts, so a crash or a time limit never costs more than the segment in
    progress, and a rerun skips everything already done.

    max_hours          stop cleanly once the *next* segment would not fit in the
                       budget (a Kaggle session lasts 12 h).
    continue_on_error  log a failed segment and carry on instead of aborting.
    after_segment      callback(n_done, n_todo, failed) after every segment,
                       e.g. to rewrite a progress index.
    """
    import time

    t_start = time.time()
    files = s01_fetch.list_files(cfg, visits)
    key = stage_key(cfg)
    todo = [f for f in files if not is_cached(stamp_path(cfg, f), key)]
    log.info("%d segments total, %d already calibrated, %d to do (%.1f GB to download)",
             len(files), len(files) - len(todo), len(todo),
             sum(f.size_gb for f in todo if not Path(f.local_path).exists()))
    if max_segments is not None:
        todo = todo[:max_segments]
    if dry_run:
        return [process_segment(cfg, rf, dry_run=True) for rf in todo]
    # Download ahead: while segment i is being calibrated, segment i+1 is fetched in a
    # background thread, so the network and the CPU are both busy. At most two raw
    # files are on disk at any time. (With attached inputs the "download" returns at once.)
    from concurrent.futures import ThreadPoolExecutor

    min_free = cfg["source"].get("min_free_gb", 8.0)
    done, failed, longest = [], [], 0.0
    with ThreadPoolExecutor(max_workers=1) as pool:
        nxt = pool.submit(s01_fetch.download, todo[0], min_free) if todo else None
        for i, rf in enumerate(todo, 1):
            elapsed_h = (time.time() - t_start) / 3600.0
            if max_hours is not None and elapsed_h + 1.5 * longest / 3600.0 > max_hours:
                log.warning("time budget of %.1f h reached after %.1f h: %d segments left, rerun to continue",
                            max_hours, elapsed_h, len(todo) - i + 1)
                break
            t_seg = time.time()
            try:
                nxt.result()                              # raises here if the download failed
                nxt = pool.submit(s01_fetch.download, todo[i], min_free) if i < len(todo) else None
                log.info("[%d/%d] %s %s", i, len(todo), rf.visit, rf.filename)
                done.append(process_segment(cfg, rf))
            except Exception as e:                        # noqa: BLE001 - reported, optionally re-raised
                if not continue_on_error:
                    raise
                log.error("FAILED %s %s: %s: %s", rf.visit, rf.filename, type(e).__name__, e)
                failed.append(rf.filename)
                if nxt is None or nxt.done():
                    nxt = pool.submit(s01_fetch.download, todo[i], min_free) if i < len(todo) else None
            longest = max(longest, time.time() - t_seg)
            if after_segment is not None:
                after_segment(len(done), len(todo), failed)
    if failed:
        log.error("%d segments failed: %s", len(failed), ", ".join(failed))
    return done


def load_visit_stamps(cfg: dict, visit: str) -> dict:
    """Concatenate all cached segment stamps of a visit, in time order."""
    files = s01_fetch.list_files(cfg, [visit])
    key = stage_key(cfg)
    parts, missing = [], []
    for rf in files:
        p = stamp_path(cfg, rf)
        if is_cached(p, key):
            parts.append(np.load(p, allow_pickle=False))
        else:
            missing.append(rf.filename)
    if missing:
        raise FileNotFoundError(f"{visit}: {len(missing)} segments not calibrated yet, e.g. {missing[0]}")
    out = {k: np.concatenate([p[k] for p in parts]) for k in ("time", "sci", "err", "bad")}
    out["origin"] = parts[0]["origin"]
    out["centre"] = parts[0]["centre"]
    out["median_frame"] = parts[0]["median_frame"]
    out["pixar_sr"] = float(parts[0]["pixar_sr"])
    out["unit"] = str(parts[0]["unit"])
    out["segment_starts"] = np.cumsum([0] + [len(p["time"]) for p in parts])[:-1]
    out["n_segments"] = len(parts)
    return out

In [ ]:
%%writefile /kaggle/working/rockyeye_src/configs/gj3929b.yaml
# RockyEye config — real JWST/MIRI F1500W eclipses of GJ 3929 b (Rocky Worlds DDT, program 9235).
# Do not run until the simulated-data validation (configs/lhs1140b_sim.yaml) has passed.
name: gj3929b
requires_validation: configs/lhs1140b_sim.yaml   # real data stay locked until this passes
target: GJ 3929 b

source:
  kind: kaggle             # the challenge's frozen copy of the program-9235 uncal files (same as MAST)
  dataset: stsci/rocky-worlds-gj-3929b-observations
  manifest: data/manifests/gj3929b/manifest.csv
  # Eclipse 4 was split in two exposures (frame-count limit) with a 16 min gap near
  # mid-eclipse: observation 4 and observation 5. They become separate fit units.
  visit_from_obsnum: {4: eclipse4a, 5: eclipse4b}
  min_free_gb: 8
  total_gb_expected: 32

paths:
  data: data/gj3929b
  products: products/gj3929b
  crds: data/crds_cache

# Fit units. Each has its own systematics model; eclipse 4 is two units sharing one depth.
visits: [eclipse1, eclipse2, eclipse3, eclipse4a, eclipse4b]
eclipses:
  eclipse1: [eclipse1]
  eclipse2: [eclipse2]
  eclipse3: [eclipse3]
  eclipse4: [eclipse4a, eclipse4b]

keep: {raw: false, rateints: false, calints: false}

stage1:
  chunk_ints: 50          # integrations calibrated at once (bounds memory on a 16 GB laptop)
  steps:
    jump: {rejection_threshold: 6.0, minimum_sigclip_groups: 1000000}
    ramp_fit: {maximum_cores: half}
stage2:
  steps:
    resample: {skip: true}

stamp:
  half_size: 40

photometry:
  aperture_radii: [3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 8.0]
  annulus: [16.0, 30.0]
  centroid_box: 5
  outlier_sigma: 5.0
  gain: 3.1

# Orbit: Xue et al. 2025 (arXiv:2508.12516, ApJL), Table 1 (joint TESS + RV + ground fit).
# Star: Kemmer et al. 2022 (A&A, arXiv:2202.00970), Table 2: Teff = 3369 +/- 51 K,
#   R = 0.315 +/- 0.010 Rsun, log g = 4.84, d = 15.830 +/- 0.006 pc (Gaia EDR3 parallax
#   63.173 +/- 0.020 mas). These are the values Xue+25 adopted (their T_max = 737 +/- 14 K
#   follows from them). Checked 2026-11-13 against the ar5iv copies of both papers.
# Alternative, not used: Beard et al. 2022 (ApJ, arXiv:2207.10672), Table 2: Teff = 3384 +/- 88 K,
#   R = 0.32 +/- 0.01 Rsun, d = 15.822 +/- 0.006 pc. The Teff difference (15 K) changes
#   T_max by 3 K; the radius difference (1.6%) changes the flux-based stellar brightness by 3%.
system:
  period: 2.6162644
  t0: 2460452.8997
  a_rs: 17.05
  a_rs_err: 0.43
  inc: 89.3
  rp_rs: 0.0318
  rp_rs_err: 0.0007
  teff: 3369.0
  teff_err: 51.0
  rstar_rsun: 0.315
  rstar_rsun_err: 0.010
  dist_pc: 15.830
  dist_pc_err: 0.006

fit:
  trim_start: 500
  bin_size: 4
  # Centroid (x, y) decorrelation is the floor of every candidate; BIC chooses the extras.
  # Why: on synthetic data with pointing drift, dropping x,y kept the depth unbiased on
  # average but tripled its scatter (77 vs 24 ppm, 10 realisations) while the quoted error
  # stayed ~35 ppm, and "simplest within dBIC 2" dropped them in 3 of 6 visits. Without
  # drift, keeping x,y costs nothing (23 vs 23 ppm). See REPORT.md section 5.1.
  candidate_models: [linear+xy, poly2+xy, exp+xy, exp+xyw, exp+xywbg, exp2+xy]
  priors:
    depth_ppm: [-500, 1000]
    dt_min: [-120, 60]
    tau_hours: [0.02, 8.0]
  shared_time: true
  mcmc: {n_walkers: 64, max_steps: 60000, check_every: 3000, tau_multiple: 50, seed: 42}
  extra_joint_fits:
    joint_12: [eclipse1, eclipse2]     # the two visits analysed by Xue et al. 2025

injection:
  depths_ppm: [0, 50, 100, 150, 200, 300]
  n_trials: 200
  detection_sigma: 3.0
  n_dt_grid: 37

physics:
  seed: 11
  n_draws: 20000
  primary_stellar_model: measured_flux
  # Aperture/sky radii and correction for 70% encircled energy: CRDS jwst_miri_apcorr_0014.fits (F1500W).
  flux_calibration: {radius: 5.6944, skyin: 8.6297, skyout: 11.4520, apcorr: 1.4974, systematic_frac: 0.03, apply_to: measured_flux}
  stellar_models:
    measured_flux: {mode: flux, flux_mjy: 19.19, flux_mjy_err: 0.6}   # replaced by our own measurement in stage 07
    blackbody_teff: {mode: blackbody}
    sphinx_literature: {mode: flux, flux_mjy: 18.8, flux_mjy_err: 0.6}  # SPHINX value quoted by Connors+ 2026
  reference_scenario: bare rock (A=0)
  scenarios:
    bare rock (A=0): {albedo: 0.0, eps: 0.0}
    bare rock (A=0.1): {albedo: 0.1, eps: 0.0}
    partial redistribution (eps=0.5): {albedo: 0.0, eps: 0.5}
    full redistribution (A=0): {albedo: 0.0, eps: 1.0}
    full redistribution (A=0.3): {albedo: 0.3, eps: 1.0}
    "lit: Venus-like 100 mbar": {depth_ppm: 36.1, depth_err_ppm: 2.4}      # Connors+ 2026 Table 4
    "lit: O2 + 1% CO2 100 mbar": {depth_ppm: 37.1, depth_err_ppm: 2.4}
    "lit: thin O2 + 0.1% CO2 30 mbar": {depth_ppm: 91.0, depth_err_ppm: 6.0}

# Published results to benchmark against (ppm).
# "RWDDT HLSP": the Rocky Worlds team's own archive products (MAST HLSP, DOI 10.17909/qsyr-ny68,
# Eureka!-based; files in data/benchmarks/gj3929b_hlsp/). Single-eclipse entries are v1.0.1
# eclipse catalogs; the joint entry is "checkpoint02" (v1.0), which combines all four eclipses.
published:
  - {ref: "RWDDT HLSP", label: eclipse1, depth_ppm: -9.1, err_ppm: 95.5}
  - {ref: "RWDDT HLSP", label: eclipse2, depth_ppm: 16.2, err_ppm: 78.1}
  - {ref: "RWDDT HLSP", label: eclipse3, depth_ppm: 174.0, err_ppm: 37.4}
  - {ref: "RWDDT HLSP", label: eclipse4, depth_ppm: 190.7, err_ppm: 130.9}
  - {ref: "RWDDT HLSP", label: joint, depth_ppm: 142.5, err_ppm: 18.6}
  - {ref: "Xue+25", label: eclipse1, depth_ppm: 177, err_ppm: 46}
  - {ref: "Xue+25", label: eclipse2, depth_ppm: 143, err_ppm: 35}
  - {ref: "Xue+25", label: joint_12, depth_ppm: 160, err_ppm: 26}
  - {ref: "Connors+26 (FN-PCA)", label: eclipse1, depth_ppm: 190, err_ppm: 40}
  - {ref: "Connors+26 (FN-PCA)", label: eclipse2, depth_ppm: 155, err_ppm: 34}
  - {ref: "Connors+26 (FN-PCA)", label: eclipse3, depth_ppm: 60, err_ppm: 34}
  - {ref: "Connors+26 (FN-PCA)", label: eclipse4, depth_ppm: 97, err_ppm: 78}
  - {ref: "Connors+26 (FN-PCA)", label: joint, depth_ppm: 118, err_ppm: 22}
  - {ref: "Connors+26 (FN-PCA)", label: joint_12, depth_ppm: 172, err_ppm: 26}
  - {ref: "Connors+26 (poly3)", label: eclipse1, depth_ppm: 115, err_ppm: 53}
  - {ref: "Connors+26 (poly3)", label: eclipse2, depth_ppm: 183, err_ppm: 40}
  - {ref: "Connors+26 (poly3)", label: eclipse3, depth_ppm: 92, err_ppm: 39}
  - {ref: "Connors+26 (poly3)", label: eclipse4, depth_ppm: 129, err_ppm: 89}
  - {ref: "Connors+26 (poly3)", label: joint, depth_ppm: 126, err_ppm: 25}
published_temperature:
  - {ref: "Xue+25 (SPHINX)", T_b: 782, err: 79}
  - {ref: "Xue+25 (observed flux)", T_b: 745, err: 76}
  - {ref: "Connors+26", T_b: 641, err: 62}

## Run stages 01–02

In [ ]:
import csv, glob, json, os, shutil, sys, time

os.environ["CRDS_PATH"] = "/kaggle/temp/crds_cache"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
sys.path.insert(0, "/kaggle/working/rockyeye_src")

import numpy as np
from rockyeye.common import load_config, setup_logging
from rockyeye import s01_fetch, s02_calibrate

setup_logging("/kaggle/working/stamps/calibrate.log")
T0 = time.time()


def find_input_root(cfg):
    """Folder of the attached dataset: the one whose manifest.csv lists this target's data type."""
    want = "simulation" if "sim" in cfg["name"] else "real_observation"
    for man in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
        with open(man) as f:
            row = next(csv.DictReader(f), None)
        if row and row.get("data_type") == want:
            return os.path.dirname(man)
    raise FileNotFoundError(f"dataset {cfg['source']['dataset']} is not attached (Add Input)")


def restore_previous(out_dir, name):
    """Copy stamps from an attached earlier run so they are not recomputed."""
    n = 0
    for p in glob.glob(f"/kaggle/input/**/stamps/{name}/stage02/**/*", recursive=True):
        if os.path.isfile(p):
            rel = p.split(f"/stamps/{name}/stage02/", 1)[1]
            dest = os.path.join(out_dir, "stage02", rel)
            if not os.path.exists(dest):
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copyfile(p, dest)
                n += 1
    return n


name = TARGET
cfg = load_config(f"/kaggle/working/rockyeye_src/configs/{name}.yaml")
cfg["source"]["local_root"] = find_input_root(cfg)          # read the attached files in place
cfg["paths"] = {"data": f"/kaggle/temp/rockyeye/{name}",     # scratch: chunks, rateints, calints
                "products": f"/kaggle/working/stamps/{name}", "crds": "/kaggle/temp/crds_cache"}
cfg["keep"] = {"raw": True, "rateints": False, "calints": False}
print(f"=== {name}: input {cfg['source']['local_root']}")
print("restored from a previous run:", restore_previous(cfg["paths"]["products"], name), "files")
print(s01_fetch.format_plan(s01_fetch.plan(cfg, VISITS)))
files = s01_fetch.list_files(cfg, VISITS)
key = s02_calibrate.stage_key(cfg)


FAILED = []


def write_index(n_done=None, n_todo=None, failed=None):
    """Rewrite the progress index; called after every segment so it is never stale."""
    if failed is not None:
        FAILED[:] = failed
    done = [f for f in files if s02_calibrate.is_cached(s02_calibrate.stamp_path(cfg, f), key)]
    index = {name: {"stage_key": key, "segments_total": len(files), "segments_done": len(done),
                    "hours_elapsed": round((time.time() - T0) / 3600.0, 2), "failed": list(FAILED),
                    "missing": [f.filename for f in files if f not in done]}}
    tmp = "/kaggle/working/stamps/stamps_index.json.tmp"
    with open(tmp, "w") as fh:
        json.dump(index, fh, indent=1)
    os.replace(tmp, "/kaggle/working/stamps/stamps_index.json")
    return index[name]


write_index()
s02_calibrate.process(cfg, VISITS, max_hours=MAX_HOURS - (time.time() - T0) / 3600.0,
                      continue_on_error=True, after_segment=write_index)
status = write_index()
shutil.rmtree("/kaggle/temp/rockyeye", ignore_errors=True)
print(json.dumps({k: v for k, v in status.items() if k != "missing"}, indent=1))
print("ALL DONE" if status["segments_done"] == status["segments_total"]
      else "INCOMPLETE: start a new version with this version's output attached as input to continue")

## Quick look

In [ ]:
# Quick look: crude light curve per visit (sum of the central 11x11 px, sky from the stamp edge).
# Only a sanity check that stamps contain a star and look stable; real photometry is stage 03.
import matplotlib.pyplot as plt

for name in [TARGET]:
    visits = sorted(os.listdir(f"/kaggle/working/stamps/{name}/stage02")) if os.path.isdir(f"/kaggle/working/stamps/{name}/stage02") else []
    if not visits:
        continue
    fig, axes = plt.subplots(len(visits), 1, figsize=(9, 1.8 * len(visits)), squeeze=False)
    for ax, v in zip(axes[:, 0], visits):
        t, f = [], []
        for p in sorted(glob.glob(f"/kaggle/working/stamps/{name}/stage02/{v}/*_stamp.npz")):
            z = np.load(p)
            s = z["sci"]
            c = s.shape[1] // 2
            sky = np.nanmedian(s[:, :6, :], axis=(1, 2))
            f.append(np.nansum(s[:, c - 5:c + 6, c - 5:c + 6] - sky[:, None, None], axis=(1, 2)))
            t.append(z["time"])
        if t:
            t, f = np.concatenate(t), np.concatenate(f)
            ax.plot((t - t[0]) * 24, f / np.nanmedian(f), ".", ms=1)
            ax.set_ylim(0.99, 1.01)
        ax.set_ylabel(v)
    axes[-1, 0].set_xlabel("hours since start")
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()